# FCR：1.6 m 水质数据转 icecore 风格宽表

输入：`dataset/fcr/ori/eco-KGML_model_runs.csv`。仅处理五个水质变量，不读取驱动表。

每张输出 CSV 的行是日、列是模拟情景（`FCR_0001` 等），不是实际地理站点。
为兼容现有 icecore loader，第一列仍叫 `year`，但含义是从首个水质日期起、从 0 开始的**整数日序号**，不是公历年。
五张变量 CSV 保存到 `dataset/fcr/`。
`preprocessing/date_mapping.csv` 保存日序号与真实日期；`preprocessing/scenario_mapping.csv` 保存列名与原始情景编号。
本 notebook 位于 `dataset/fcr/preprocessing/`。summary 仅在单元格输出中显示，不生成文件。

不插值、不归一化、不去噪、不划分训练集。缺失保留为空；标记含义未经确认，因此只统计、不按 Flag 擅自删除数值。
发现同一日期与情景的重复记录会停止，避免默默求均值。五张表使用完全相同的日历和情景顺序。
此 notebook 仅准备数据；训练时仍需重新配置 FCR 的站点特征与划分，不能沿用冰芯元数据。


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# 支持从项目根目录、dataset/fcr 或本 notebook 的 preprocessing 目录运行。
BASE = next((p.resolve() for p in
             (Path.cwd(), Path.cwd().parent, Path.cwd() / 'dataset/fcr')
             if (p / 'ori/eco-KGML_model_runs.csv').is_file()), None)
if BASE is None:
    raise FileNotFoundError('请从项目根目录、dataset/fcr 或 dataset/fcr/preprocessing 运行。')
SOURCE = BASE / 'ori/eco-KGML_model_runs.csv'
PREPROCESSING = BASE / 'preprocessing'
PREPROCESSING.mkdir(exist_ok=True)
DEPTH = 1.6
VARIABLES = ['WaterTemp_C', 'SRP_ugL', 'DIN_ugL', 'LightAttenuation_Kd', 'Chla_ugL']
FLAGS = ['Flag_' + v for v in VARIABLES]
print('输入:', SOURCE.resolve())
print('宽表输出目录:', BASE)
print('映射表输出目录:', PREPROCESSING)


输入: /Users/liulei/Desktop/SEMPO_copy0707/dataset/fcr/ori/eco-KGML_model_runs.csv
宽表输出目录: /Users/liulei/Desktop/SEMPO_copy0707/dataset/fcr
映射表输出目录: /Users/liulei/Desktop/SEMPO_copy0707/dataset/fcr/preprocessing


## 1. 分块读取，只保留 1.6 m
原文件约 1.3 GB；分块读取避免一次载入全部深度。保留原数值精度。

In [2]:
columns = ['Lake', 'Site', 'DataType', 'DateTime', 'Depth_m', 'ModelRunType'] + VARIABLES + FLAGS
parts = []
source_rows = 0
for chunk in pd.read_csv(SOURCE, usecols=columns, chunksize=250_000, float_precision='round_trip'):
    source_rows += len(chunk)
    keep = np.isclose(chunk['Depth_m'], DEPTH, rtol=0, atol=1e-8)
    parts.append(chunk.loc[keep].copy())
data = pd.concat(parts, ignore_index=True)
del parts
if data.empty:
    raise ValueError('指定深度没有记录。')
data['DateTime'] = pd.to_datetime(data['DateTime'], errors='raise')
assert data['DateTime'].eq(data['DateTime'].dt.normalize()).all(), '含日内时间，需要明确聚合规则。'
assert data['ModelRunType'].notna().all()
assert (data['ModelRunType'] % 1 == 0).all()
data['ModelRunType'] = data['ModelRunType'].astype('int64')
assert data['Lake'].nunique(dropna=False) == 1, '发现多个湖，不能只按情景编号合并。'
assert data['Site'].nunique(dropna=False) == 1, '发现多个实体站点，需要扩展主键。'
duplicates = data.duplicated(['DateTime', 'ModelRunType'], keep=False)
if duplicates.any():
    raise ValueError(f'发现 {duplicates.sum()} 条重复情景/日期记录，请先检查，不自动聚合。')
assert not np.isinf(data[VARIABLES].to_numpy(dtype=float)).any(), '存在无穷值。'
print(f'原表行数: {source_rows:,}; 1.6 m 行数: {len(data):,}')
print('Lake/Site/DataType:', data[['Lake', 'Site', 'DataType']].drop_duplicates().to_dict('records'))
print(data[['DateTime', 'ModelRunType'] + VARIABLES].head().to_string(index=False))
print('质量标记取值计数（不据此过滤）:')
for flag in FLAGS:
    print(flag, data[flag].value_counts(dropna=False).to_dict())


原表行数: 10,437,000; 1.6 m 行数: 1,491,000
Lake/Site/DataType: [{'Lake': 'FCR', 'Site': 50, 'DataType': 'modeled'}]
  DateTime  ModelRunType  WaterTemp_C  SRP_ugL   DIN_ugL  LightAttenuation_Kd  Chla_ugL
2016-12-02             1     8.411637 6.365287  8.464486             0.550946  1.487897
2016-12-03             1     7.959375 6.363022  9.042624             0.548863  1.323383
2016-12-04             1     7.587172 6.402085  9.522326             0.547502  1.187736
2016-12-05             1     7.389265 6.427584  9.942351             0.546131  1.075963
2016-12-06             1     7.092683 6.464097 10.326255             0.545232  0.990790
质量标记取值计数（不据此过滤）:
Flag_WaterTemp_C {0: 1491000}
Flag_SRP_ugL {0: 1491000}
Flag_DIN_ugL {0: 1491000}
Flag_LightAttenuation_Kd {0: 1491000}
Flag_Chla_ugL {0: 1491000}


## 2. 建立共享日历和情景列
即使原始数据某一天缺测，也保留该日期，避免将不连续的日期压缩成相邻时间步。

In [3]:
dates = pd.date_range(data['DateTime'].min(), data['DateTime'].max(), freq='D')
scenario_ids = sorted(data['ModelRunType'].unique())
site_columns = [f'FCR_{sid:04d}' for sid in scenario_ids]
day_index = np.arange(len(dates), dtype=np.int64)
date_mapping = pd.DataFrame({'year': day_index, 'DateTime': dates.strftime('%Y-%m-%d')})
scenario_mapping = pd.DataFrame({'site': site_columns, 'ModelRunType': scenario_ids})
print(f'情景数: {len(scenario_ids)}; 日期数: {len(dates)}')
print(f'日期范围: {dates[0].date()} 至 {dates[-1].date()}')
print(f'情景编号范围: {scenario_ids[0]} 至 {scenario_ids[-1]}')
print('缺失的情景-日期组合数:', len(dates) * len(scenario_ids) - len(data))


情景数: 1000; 日期数: 1491
日期范围: 2016-12-02 至 2020-12-31
情景编号范围: 1 至 1000
缺失的情景-日期组合数: 0


## 3. 导出五张宽表与映射表
文件名直接使用原变量名。每次重跑会更新本 notebook 生成的同名文件。

In [ ]:
summaries = []
for variable in VARIABLES:
    wide = data.pivot(index='DateTime', columns='ModelRunType', values=variable)
    wide = wide.reindex(index=dates, columns=scenario_ids)
    summaries.append({
        'variable': variable, 'depth_m': DEPTH, 'days': len(dates), 'scenarios': len(scenario_ids),
        'missing_cells': int(wide.isna().sum().sum()),
        'min': wide.min().min(), 'max': wide.max().max(),
    })
    wide.columns = site_columns
    wide.index = pd.Index(day_index, name='year')
    wide.to_csv(BASE / f'{variable}.csv', na_rep='')
    print(f'{variable}.csv: {wide.shape[0]} 行 × {wide.shape[1] + 1} 列（含 year）')
date_mapping.to_csv(PREPROCESSING / 'date_mapping.csv', index=False)
scenario_mapping.to_csv(PREPROCESSING / 'scenario_mapping.csv', index=False)
summary = pd.DataFrame(summaries)
# summary 只在 notebook 中显示，不生成 CSV。
print(summary.to_string(index=False))


WaterTemp_C.csv: 1491 行 × 1001 列（含 year）
SRP_ugL.csv: 1491 行 × 1001 列（含 year）
DIN_ugL.csv: 1491 行 × 1001 列（含 year）
LightAttenuation_Kd.csv: 1491 行 × 1001 列（含 year）
Chla_ugL.csv: 1491 行 × 1001 列（含 year）
           variable  depth_m  days  scenarios  missing_cells      min        max
        WaterTemp_C      1.6  1491       1000              0 1.819621  29.575865
            SRP_ugL      1.6  1491       1000              0 0.080054  34.569813
            DIN_ugL      1.6  1491       1000              0 0.069977 191.308475
LightAttenuation_Kd      1.6  1491       1000              0 0.455403   1.914340
           Chla_ugL      1.6  1491       1000              0 0.051000 266.514215


## 4. 回读校验
逐个文件检查时间轴、列顺序、缺失位置以及所有水质数值，确认宽表没有行列错位或改变数据。

In [5]:
for variable in VARIABLES:
    saved = pd.read_csv(BASE / f'{variable}.csv', float_precision='round_trip')
    assert saved.columns.tolist() == ['year'] + site_columns
    np.testing.assert_array_equal(saved['year'].to_numpy(), day_index)
    expected = data.pivot(index='DateTime', columns='ModelRunType', values=variable).reindex(
        index=dates, columns=scenario_ids).to_numpy(dtype=float)
    np.testing.assert_allclose(saved.iloc[:, 1:].to_numpy(dtype=float), expected,
                               rtol=0, atol=0, equal_nan=True)
    print(variable, '全量回读一致')
print('宽表预览（第一个变量，前 3 天、前 4 个情景）:')
print(pd.read_csv(BASE / f'{VARIABLES[0]}.csv', nrows=3).iloc[:, :5].to_string(index=False))


WaterTemp_C 全量回读一致
SRP_ugL 全量回读一致
DIN_ugL 全量回读一致
LightAttenuation_Kd 全量回读一致
Chla_ugL 全量回读一致
宽表预览（第一个变量，前 3 天、前 4 个情景）:
 year  FCR_0001  FCR_0002  FCR_0003  FCR_0004
    0  8.411637  8.412027  8.411775  8.411942
    1  7.959375  7.959502  7.959201  7.959373
    2  7.587172  7.587805  7.587642  7.587860
